# Procurement ML — four assistive models for the RAG assistant

Four supervised models that the GenAI layer calls when a question needs a
*number* rather than a passage. The split of labour: retrieval and phrasing stay
with the LLM, while anything involving budgets, survival curves, or approval
odds goes to a model fitted on procurement history.

| # | Use case | Task | Target |
|---|---|---|---|
| UC1 | Role-to-laptop suitability | Binary classification | `target_uc1_is_match` |
| UC2 | Lifespan / time to failure | Regression | `target_uc2_months_to_failure` |
| UC3 | 3-year operating cost | Regression | `target_uc3_opex_idr` |
| UC4 | Procurement approval | Binary classification (imbalanced) | `target_uc4_is_approved` |

Data comes from `syntetic_data_complete.py`. Its labels are sampled from latent
scores rather than written as if/else rules, so the metrics below reflect a real
decision surface with irreducible Bayes error — a rule-based label would just be
a lookup table that any tree recovers at ~100% accuracy.

**A note on what this is.** The dataset is synthetic. These numbers demonstrate a
working pipeline; they are not evidence about real procurement outcomes. Swap in
historical data and the same notebook re-fits unchanged.

## 1. Setup and data

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, f1_score, mean_absolute_error, precision_recall_curve,
    r2_score, roc_auc_score, root_mean_squared_error,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier, XGBRegressor

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

from syntetic_data_complete import (
    CATEGORICAL_FEATURES, DEPARTMENTS, FEATURE_COLUMNS, ID_COLUMNS,
    NUMERIC_FEATURES, TARGET_COLUMNS, generate_procurement_ml_data,
)

df = generate_procurement_ml_data(6000)
print("shape:", df.shape)
df.head(3)

In [ ]:
print("Label summary")
print(f"  UC1 suitability  : {df.target_uc1_is_match.mean():.1%} positive")
print(f"  UC4 approval     : {df.target_uc4_is_approved.mean():.1%} positive "
      f"-> {1 - df.target_uc4_is_approved.mean():.0%} minority class")
print(f"  UC2 lifespan     : mean {df.target_uc2_months_to_failure.mean():.1f} months, "
      f"sd {df.target_uc2_months_to_failure.std():.1f}")
print(f"  UC3 operating    : mean {df.target_uc3_opex_idr.mean():,.0f} IDR "
      f"(of {df.target_uc3_tco_idr.mean():,.0f} total TCO)")

fig, axes = plt.subplots(1, 4, figsize=(17, 3.2))
df.target_uc1_is_match.value_counts().sort_index().plot.bar(ax=axes[0], title="UC1 suitability", rot=0)
df.target_uc2_months_to_failure.plot.hist(ax=axes[1], bins=40, title="UC2 months to failure")
(df.target_uc3_opex_idr / 1e6).plot.hist(ax=axes[2], bins=40, title="UC3 operating cost (million IDR)")
df.target_uc4_is_approved.value_counts().sort_index().plot.bar(ax=axes[3], title="UC4 approval", rot=0)
plt.tight_layout()
plt.show()

## 2. Feature engineering

Two kinds of derived feature, both computed only from inputs — never from a
label:

- **Commercial ratios.** A tree can only split `requested_unit_price` at fixed
  thresholds; what actually drives approval is price *relative* to the
  historical average, the policy cap, and the remaining budget. Giving the model
  the ratio directly saves it from approximating a division with a staircase of
  splits.
- **Spec-adequacy.** `spec_gap` compares the machine's compute capability
  against what the role needs, on one scale. This is the feature that lets the
  assistant say *"32 GB is overkill for this role"* — a positive gap means
  over-provisioned, negative means under-provisioned. Encoding it as one signed
  number is far easier to learn than the `department × ram_gb` interaction it
  replaces.

In [ ]:
DEPT_COMPUTE_NEED = {name: cfg["compute_need"] for name, cfg in DEPARTMENTS.items()}


def engineer_features(frame):
    """Derive ratio and adequacy features. Inputs only -- no label is read here."""
    out = frame.copy()

    # --- commercial position of the request
    out["price_premium_ratio"] = (
        (out.requested_unit_price - out.historical_avg_price) / out.historical_avg_price
    )
    out["budget_utilisation"] = out.total_amount / out.dept_budget_remaining
    out["cap_overrun_ratio"] = (
        (out.requested_unit_price - out.dept_policy_cap) / out.dept_policy_cap
    )

    # --- spec adequacy: how far the machine sits from what the role needs
    capability = out.compute_tier + 0.55 * np.log2(out.ram_gb.clip(lower=1) / 8.0)
    out["spec_gap"] = capability - out.department.map(DEPT_COMPUTE_NEED)
    out["is_overspec"] = (out.spec_gap > 0.75).astype(int)
    out["is_underspec"] = (out.spec_gap < -0.75).astype(int)

    # --- value density
    out["price_per_ram_gb"] = out.requested_unit_price / out.ram_gb
    out["price_per_storage_gb"] = out.requested_unit_price / out.storage_gb

    # --- usage intensity
    out["tickets_per_travel_day"] = out.it_tickets_last_year / (out.travel_days_per_month + 1.0)
    out["wear_exposure"] = out.travel_days_per_month * (1.0 - out.build_quality)

    return out


ENGINEERED = [
    "price_premium_ratio", "budget_utilisation", "cap_overrun_ratio",
    "spec_gap", "is_overspec", "is_underspec",
    "price_per_ram_gb", "price_per_storage_gb",
    "tickets_per_travel_day", "wear_exposure",
]

df_fe = engineer_features(df)
NUMERIC_ALL = NUMERIC_FEATURES + ENGINEERED

print(f"{len(FEATURE_COLUMNS)} raw features + {len(ENGINEERED)} engineered "
      f"= {len(CATEGORICAL_FEATURES) + len(NUMERIC_ALL)} total")
df_fe[ENGINEERED].describe().T[["mean", "std", "min", "max"]].round(3)

## 3. Leakage guard and preprocessing

All four labels sit in one table, which makes cross-target leakage the easiest
mistake available: feeding `target_uc2_months_to_failure` into the UC3 model
would produce a spectacular R² and a useless model, because at prediction time
nobody knows how long the laptop will last. `build_matrix` below is the single
place features are assembled, and it drops **every** `target_*` column
regardless of which use case is being fitted.

**On scaling.** Random forests and gradient-boosted trees split on rank order, so
standardising inputs changes nothing for them — including a scaler purely for
their benefit would be cargo cult. It earns its place here because each task also
fits a **regularised linear baseline** (logistic / ridge), where the L2 penalty
is scale-sensitive and unscaled inputs spanning 8 GB to 900,000,000 IDR would let
the largest-magnitude column dominate the penalty. The `ColumnTransformer` applies
it to the linear models, and the tree models simply ignore its effect.

In [ ]:
def build_matrix(frame):
    """Assemble X, dropping every target so no use case can see another's label."""
    banned = set(TARGET_COLUMNS) | set(ID_COLUMNS)
    cols = [c for c in CATEGORICAL_FEATURES + NUMERIC_ALL if c not in banned]

    leaked = banned.intersection(cols)
    if leaked:
        raise AssertionError(f"target/id column leaked into features: {leaked}")

    return frame[cols]


X = build_matrix(df_fe)
print(f"X: {X.shape[0]} rows x {X.shape[1]} features")
print("no target_* column present:", not any(c.startswith("target_") for c in X.columns))


def make_preprocessor(scale_numeric):
    """scale_numeric=True for linear models; False for trees, where it is a no-op."""
    numeric_step = StandardScaler() if scale_numeric else "passthrough"
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
        ("num", numeric_step, [c for c in NUMERIC_ALL if c in X.columns]),
    ])

## 4. Shared training and evaluation helpers

One tuning routine and one metric report per task type, so the four use cases
stay comparable. `RandomizedSearchCV` rather than an exhaustive grid: the search
spaces below have thousands of combinations and random sampling reaches
comparable optima for a fraction of the fits.

In [ ]:
# 12 draws x 3 folds = 36 fits per search. Raising these buys very little on a
# dataset this size and turns a two-minute notebook into a forty-minute one.
N_ITER = 12
CV_FOLDS = 3

RF_CLF_SPACE = {
    "model__n_estimators": [150, 250, 400],
    "model__max_depth": [None, 8, 14, 22],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__max_features": ["sqrt", "log2", 0.4, 0.7],
}
XGB_CLF_SPACE = {
    "model__n_estimators": [200, 350, 550],
    "model__max_depth": [3, 4, 6, 8],
    "model__learning_rate": [0.03, 0.06, 0.12],
    "model__subsample": [0.7, 0.85, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0],
    "model__min_child_weight": [1, 3, 6],
    "model__reg_lambda": [0.5, 1.0, 3.0, 10.0],
}
RF_REG_SPACE = RF_CLF_SPACE
XGB_REG_SPACE = XGB_CLF_SPACE


def tune(estimator, space, X_tr, y_tr, scoring, stratified, scale_numeric):
    pipe = Pipeline([("prep", make_preprocessor(scale_numeric)), ("model", estimator)])
    cv = (StratifiedKFold(CV_FOLDS, shuffle=True, random_state=RANDOM_STATE) if stratified
          else KFold(CV_FOLDS, shuffle=True, random_state=RANDOM_STATE))
    search = RandomizedSearchCV(
        pipe, space, n_iter=N_ITER, scoring=scoring, cv=cv,
        random_state=RANDOM_STATE, n_jobs=-1, refit=True,
    )
    search.fit(X_tr, y_tr)
    return search


def report_classification(name, model, X_te, y_te, threshold=0.5):
    proba = model.predict_proba(X_te)[:, 1]
    pred = (proba >= threshold).astype(int)
    row = {
        "model": name,
        "accuracy": accuracy_score(y_te, pred),
        "f1": f1_score(y_te, pred),
        "roc_auc": roc_auc_score(y_te, proba),
        "pr_auc": average_precision_score(y_te, proba),
        "threshold": threshold,
    }
    return row, proba


def report_regression(name, model, X_te, y_te):
    pred = model.predict(X_te)
    return {
        "model": name,
        "mae": mean_absolute_error(y_te, pred),
        "rmse": root_mean_squared_error(y_te, pred),
        "r2": r2_score(y_te, pred),
    }


def show(rows, float_fmt="{:.4f}"):
    out = pd.DataFrame(rows).set_index("model")
    return out.style.format(float_fmt).background_gradient(axis=0, cmap="Blues")

## 5. UC1 — Role-to-laptop suitability (classification)

*"Is this machine a good fit for this person's role?"* The label is whether a
past deployment of this configuration to this kind of employee went without
complaint or early replacement.

This is the model behind the assistant's overspec warning: `spec_gap` should
come out as a dominant feature, and it should be signed — both too little and
too much are penalised, the latter more gently.

In [ ]:
y_uc1 = df_fe.target_uc1_is_match
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_uc1, test_size=0.2, random_state=RANDOM_STATE, stratify=y_uc1,
)

uc1_results = []

uc1_lr = tune(
    LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    {"model__C": [0.01, 0.1, 1.0, 10.0]},
    X_tr, y_tr, "roc_auc", stratified=True, scale_numeric=True,
)
uc1_results.append(report_classification("LogisticRegression (scaled)", uc1_lr.best_estimator_, X_te, y_te)[0])

uc1_rf = tune(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    RF_CLF_SPACE, X_tr, y_tr, "roc_auc", stratified=True, scale_numeric=False,
)
uc1_results.append(report_classification("RandomForest", uc1_rf.best_estimator_, X_te, y_te)[0])

uc1_xgb = tune(
    XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss", tree_method="hist"),
    XGB_CLF_SPACE, X_tr, y_tr, "roc_auc", stratified=True, scale_numeric=False,
)
uc1_results.append(report_classification("XGBoost", uc1_xgb.best_estimator_, X_te, y_te)[0])

print("best RF params :", {k.replace('model__', ''): v for k, v in uc1_rf.best_params_.items()})
print("best XGB params:", {k.replace('model__', ''): v for k, v in uc1_xgb.best_params_.items()})
show(uc1_results)

In [ ]:
uc1_best = uc1_xgb.best_estimator_
pred = uc1_best.predict(X_te)
print(classification_report(y_te, pred, target_names=["poor fit", "good fit"], digits=3))
print("confusion matrix (rows = actual):")
print(pd.DataFrame(
    confusion_matrix(y_te, pred),
    index=["actual poor", "actual good"], columns=["pred poor", "pred good"],
))

## 6. UC4 — Approval prediction (imbalanced classification)

About 30% of requests are rejected. That imbalance is mild but real, and it
changes how the model must be fitted and read:

- **Class weighting, not resampling.** `class_weight="balanced"` (RF) and
  `scale_pos_weight` (XGBoost) reweight the loss directly. For tree ensembles
  this is preferable to SMOTE, which synthesises points by interpolating between
  neighbours — meaningless for the one-hot columns here, where an interpolated
  0.5 corresponds to no real department.
- **PR-AUC over ROC-AUC.** ROC-AUC is computed against the majority class and
  stays flattering under imbalance; average precision tracks performance on the
  rejections, which are the cases worth catching.
- **A tuned threshold.** 0.5 is arbitrary. The threshold is chosen on the
  training folds by maximising F1, then applied unchanged to the test set —
  picking it on the test set would be another way of leaking.

In [ ]:
y_uc4 = df_fe.target_uc4_is_approved
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_uc4, test_size=0.2, random_state=RANDOM_STATE, stratify=y_uc4,
)

neg, pos = (y_tr == 0).sum(), (y_tr == 1).sum()
scale_pos_weight = neg / pos
print(f"train balance: {pos} approved / {neg} rejected -> scale_pos_weight = {scale_pos_weight:.3f}")

uc4_results = []

uc4_lr = tune(
    LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    {"model__C": [0.01, 0.1, 1.0, 10.0]},
    X_tr, y_tr, "average_precision", stratified=True, scale_numeric=True,
)
uc4_results.append(report_classification("LogisticRegression (scaled, balanced)", uc4_lr.best_estimator_, X_te, y_te)[0])

uc4_rf = tune(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced"),
    RF_CLF_SPACE, X_tr, y_tr, "average_precision", stratified=True, scale_numeric=False,
)
uc4_results.append(report_classification("RandomForest (balanced)", uc4_rf.best_estimator_, X_te, y_te)[0])

uc4_xgb = tune(
    XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss", tree_method="hist",
                  scale_pos_weight=scale_pos_weight),
    XGB_CLF_SPACE, X_tr, y_tr, "average_precision", stratified=True, scale_numeric=False,
)
uc4_results.append(report_classification("XGBoost (scale_pos_weight)", uc4_xgb.best_estimator_, X_te, y_te)[0])

show(uc4_results)

In [ ]:
# Choose the threshold on TRAINING data only, then apply it to the test set.
uc4_best = uc4_xgb.best_estimator_
train_proba = uc4_best.predict_proba(X_tr)[:, 1]
prec, rec, thr = precision_recall_curve(y_tr, train_proba)
f1_curve = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(prec), where=(prec + rec) > 0)
best_thr = float(thr[np.argmax(f1_curve[:-1])])
print(f"threshold maximising F1 on train: {best_thr:.3f} (default would be 0.500)")

tuned_rows = [
    report_classification("XGBoost @ 0.50", uc4_best, X_te, y_te, threshold=0.5)[0],
    report_classification(f"XGBoost @ {best_thr:.2f}", uc4_best, X_te, y_te, threshold=best_thr)[0],
]
display(show(tuned_rows))

pred = (uc4_best.predict_proba(X_te)[:, 1] >= best_thr).astype(int)
print(classification_report(y_te, pred, target_names=["rejected", "approved"], digits=3))

The tuned threshold lands well below 0.5 but moves test F1 by only a fraction of
a point, in either direction depending on the seed. That is worth reporting
rather than quietly dropping: `scale_pos_weight` has already rebalanced the loss during
training, so the probabilities come out reasonably calibrated and there is not
much left for a threshold shift to recover. Threshold tuning pays off when a
model is trained *without* class weighting, or when the business cost of a false
approval differs sharply from a false rejection — at which point the threshold
should be chosen against that cost, not against F1.

## 7. UC2 — Lifespan / months to failure (regression)

Predicts how long a configuration survives in a given role before its first
major repair or replacement. Survival is right-skewed, so residuals will be
too — MAE is the metric to read here, being less distorted by the long tail than
RMSE.

Expect the **ridge baseline to match or beat both ensembles** on this one. That
is not a bug: in the generator, expected lifespan is a linear combination of
build quality, travel exposure, prior replacements, and warranty, with gamma
noise on top. A linear model is therefore correctly specified, and the trees
spend capacity approximating a straight line. It is a useful reminder that
"XGBoost won" is a result to verify per task, not a default — and a reason to
always keep a cheap baseline in the comparison.

In [ ]:
y_uc2 = df_fe.target_uc2_months_to_failure
X_tr, X_te, y_tr, y_te = train_test_split(X, y_uc2, test_size=0.2, random_state=RANDOM_STATE)

uc2_results = []

uc2_ridge = tune(Ridge(random_state=RANDOM_STATE), {"model__alpha": [0.1, 1.0, 10.0, 100.0]},
                 X_tr, y_tr, "r2", stratified=False, scale_numeric=True)
uc2_results.append(report_regression("Ridge (scaled)", uc2_ridge.best_estimator_, X_te, y_te))

uc2_rf = tune(RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
              RF_REG_SPACE, X_tr, y_tr, "r2", stratified=False, scale_numeric=False)
uc2_results.append(report_regression("RandomForest", uc2_rf.best_estimator_, X_te, y_te))

uc2_xgb = tune(XGBRegressor(random_state=RANDOM_STATE, tree_method="hist"),
               XGB_REG_SPACE, X_tr, y_tr, "r2", stratified=False, scale_numeric=False)
uc2_results.append(report_regression("XGBoost", uc2_xgb.best_estimator_, X_te, y_te))

print(f"baseline (predict the mean): MAE = {np.abs(y_te - y_tr.mean()).mean():.2f} months")
show(uc2_results, float_fmt="{:.3f}")

## 8. UC3 — 3-year cost of ownership (regression)

**Which target to predict matters more than which model to fit here.** Total TCO
is *purchase price plus operating cost*, and purchase price is a feature — so a
model predicting total TCO scores R² ≈ 0.997 by echoing an input back, with a
plain ridge regression winning. That number looks excellent and means nothing:
finance already knows the quoted price.

The useful quantity is the **operating cost** — setup, repairs net of warranty,
downtime, ecosystem overheads — which is what nobody can read off the quote.
`target_uc3_opex_idr` is that, and total TCO is recovered by adding the known
price back on. Both columns exist in the data; the cell below fits the second.

It is also genuinely harder: operating cost depends on how long the machine
lasts, which is UC2's *label* and unknown at prediction time. The model has to
infer durability from build quality, travel exposure, and ticket history instead
of being handed it.

In [ ]:
y_uc3 = df_fe.target_uc3_opex_idr
X_tr, X_te, y_tr, y_te = train_test_split(X, y_uc3, test_size=0.2, random_state=RANDOM_STATE)

uc3_results = []

uc3_ridge = tune(Ridge(random_state=RANDOM_STATE), {"model__alpha": [0.1, 1.0, 10.0, 100.0]},
                 X_tr, y_tr, "r2", stratified=False, scale_numeric=True)
uc3_results.append(report_regression("Ridge (scaled)", uc3_ridge.best_estimator_, X_te, y_te))

uc3_rf = tune(RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
              RF_REG_SPACE, X_tr, y_tr, "r2", stratified=False, scale_numeric=False)
uc3_results.append(report_regression("RandomForest", uc3_rf.best_estimator_, X_te, y_te))

uc3_xgb = tune(XGBRegressor(random_state=RANDOM_STATE, tree_method="hist"),
               XGB_REG_SPACE, X_tr, y_tr, "r2", stratified=False, scale_numeric=False)
uc3_results.append(report_regression("XGBoost", uc3_xgb.best_estimator_, X_te, y_te))

print(f"operating cost: mean {y_te.mean():,.0f} IDR, sd {y_te.std():,.0f} IDR")
print(f"baseline (predict the mean): MAE = {np.abs(y_te - y_tr.mean()).mean():,.0f} IDR")
show(uc3_results, float_fmt="{:,.3f}")

In [ ]:
# For comparison: the same pipeline against TOTAL TCO, to show why that target
# was rejected. Purchase price is a feature, so this is close to an identity.
y_total = df_fe.target_uc3_tco_idr
Xt_tr, Xt_te, yt_tr, yt_te = train_test_split(X, y_total, test_size=0.2, random_state=RANDOM_STATE)
naive = Pipeline([("prep", make_preprocessor(True)), ("model", Ridge(alpha=1.0))]).fit(Xt_tr, yt_tr)
print(f"Ridge on TOTAL TCO      -> R2 = {r2_score(yt_te, naive.predict(Xt_te)):.4f}  "
      f"(inflated: it is mostly reading requested_unit_price back)")
print(f"XGBoost on OPERATING cost -> R2 = {uc3_results[-1]['r2']:.4f}  "
      f"(the part that is actually unknown)")

## 9. What the models actually learned

A check that the fitted models rest on the features the domain says they should
— `spec_gap` for suitability, budget and price ratios for approval. If a model
scores well while leaning on something incidental, that is worth knowing before
its output reaches a user.

In [ ]:
def top_features(search, k=12):
    pipe = search.best_estimator_
    names = pipe.named_steps["prep"].get_feature_names_out()
    imp = pipe.named_steps["model"].feature_importances_
    return (pd.Series(imp, index=[n.split("__", 1)[-1] for n in names])
              .sort_values(ascending=False).head(k))


fig, axes = plt.subplots(1, 4, figsize=(19, 4.5))
for ax, (title, search) in zip(axes, [
    ("UC1 suitability", uc1_xgb), ("UC4 approval", uc4_xgb),
    ("UC2 lifespan", uc2_xgb), ("UC3 TCO", uc3_xgb),
]):
    top_features(search).sort_values().plot.barh(ax=ax, title=title)
    ax.set_xlabel("importance")
plt.tight_layout()
plt.show()

## 10. The handoff to the GenAI layer

The target interaction is the assistant turning a vague request into a scored
recommendation:

> **User** — *"Beli Macbook intel i7 dan ram 32gb buat AMGR Design"*
> **System (ML)** — *"Untuk design ram 32 gb berlebihan, walaupun lifespan lama
> dan tidak butuh banyak maintenance. Sebaiknya cari yang lain."*

`score_request` below is the seam where that happens. The LLM's job is to turn
free text into this dict and to phrase the result; the models supply the four
numbers. Fields the user has not mentioned fall back to the training median or
mode, and **the response reports which fields were guessed** — that list is what
the LLM turns into its follow-up question (*"berapa budget pengadaan?"*), rather
than quietly scoring a request built mostly from defaults.

Wiring this into `app.py` is the next step and is not done here.

In [ ]:
# Best model per task, chosen on the reported metrics rather than assuming
# XGBoost everywhere -- UC2 is the task where the ridge baseline came out ahead.
FITTED = {"uc1": uc1_xgb.best_estimator_, "uc2": uc2_ridge.best_estimator_,
          "uc3": uc3_xgb.best_estimator_, "uc4": uc4_xgb.best_estimator_}
UC4_THRESHOLD = best_thr

_defaults = {c: (df_fe[c].mode()[0] if c in CATEGORICAL_FEATURES else df_fe[c].median())
             for c in CATEGORICAL_FEATURES + NUMERIC_FEATURES}


def score_request(**fields):
    """Score a partially-specified request and say what had to be assumed."""
    unknown = set(fields) - set(_defaults)
    if unknown:
        raise KeyError(f"unknown field(s): {sorted(unknown)}")

    row = {**_defaults, **fields}
    assumed = sorted(set(_defaults) - set(fields))

    # Keep derived columns consistent with whatever the caller did supply.
    row["total_amount"] = row["requested_unit_price"] * row["quantity"]

    frame = engineer_features(pd.DataFrame([row]))
    matrix = build_matrix(frame)

    approve_proba = float(FITTED["uc4"].predict_proba(matrix)[0, 1])
    opex = float(FITTED["uc3"].predict(matrix)[0])
    return {
        "suitability_proba": float(FITTED["uc1"].predict_proba(matrix)[0, 1]),
        "expected_lifespan_months": float(FITTED["uc2"].predict(matrix)[0]),
        "expected_opex_idr": opex,
        # The model predicts only the unknown half; the quoted price is known,
        # so total TCO is just added back on.
        "expected_total_tco_idr": opex + row["requested_unit_price"] * row["quantity"],
        "approval_proba": approve_proba,
        "approval_decision": "likely approved" if approve_proba >= UC4_THRESHOLD else "likely rejected",
        "spec_gap": float(frame.spec_gap.iloc[0]),
        "assumed_fields": assumed,
    }

In [ ]:
# The example request: a 32 GB MacBook-class machine for a Design manager.
overspec = score_request(
    department="Design", seniority_level="Manager",
    laptop_brand="Apple", laptop_model="MacBook Pro 14",
    cpu="M3 Pro", gpu="Apple GPU",
    ram_gb=32, storage_gb=512, compute_tier=4,
    build_quality=0.95, portability=0.85,
    requested_unit_price=36_000_000, quantity=1,
)

# The same role, a right-sized machine.
rightsized = score_request(
    department="Design", seniority_level="Manager",
    laptop_brand="HP", laptop_model="ZBook Firefly",
    cpu="Core i7", gpu="RTX A500",
    ram_gb=16, storage_gb=512, compute_tier=3,
    build_quality=0.80, portability=0.60,
    requested_unit_price=30_000_000, quantity=1,
)

comparison = pd.DataFrame({"32 GB MacBook Pro": overspec, "16 GB ZBook Firefly": rightsized}).T
print(comparison[["suitability_proba", "spec_gap", "expected_lifespan_months",
                  "expected_opex_idr", "expected_total_tco_idr",
                  "approval_proba", "approval_decision"]].to_string())
print("\nfields the assistant would need to ask about:", overspec["assumed_fields"])

### Reading the comparison

`spec_gap` is the number the assistant translates into *"32 GB berlebihan"* — a
positive gap is capability beyond what the Design role consumes. Note that the
overspecified option can still score well on lifespan and TCO while losing on
suitability: the models disagree, and that disagreement is exactly the nuance in
the target response (*"walaupun lifespan lama dan tidak butuh banyak
maintenance"*). Handing the LLM four separate numbers rather than one verdict is
what lets it give a reasoned recommendation instead of a bare yes or no.

`assumed_fields` lists everything defaulted rather than stated. Those are the
follow-up questions the assistant should ask before treating any of this as
advice.